# 01 — Ingestion (bronze layer)

**Purpose.** Read the ten raw inputs under `data/raw/`, validate each
against its documented schema, and materialise each as a schema-faithful
parquet snapshot under `data/processed/<source>/<filename>.parquet`. No
transforms, joins, derivations, renames, or type coercions beyond what
`pd.read_csv` / `pd.read_excel` does natively. Documented preprocessing
(AWS skiprows=1, Ember column reconciliation, IM3 layer filtering,
Aqueduct sentinel handling, indicator/weight filters) is deferred to
Phase 2.

**Inputs.**

- `../data/raw/epoch/epoch_all_models.csv`
- `../data/raw/ember/ember_yearly.csv`
- `../data/raw/ember/ember_us_states.csv`
- `../data/raw/im3/im3_open_source_data_center_atlas_v2026_02_09.csv`
- `../data/raw/aqueduct/Aqueduct40_rankings_download_Y2023M07D05.xlsx` (sheets `country_baseline`, `province_baseline`)
- `../data/raw/jegham/DataSnapshotOct26.csv`
- `../data/raw/jegham/artificialanalysis_environmental.csv`
- `../data/raw/jegham/Monthly_LLM_Environmental_Footprint.csv`
- `../data/raw/jegham/AWS_Env_Multipliers.csv`
- `../data/raw/jegham/Microsoft_Env_Multipliers.csv`

**Outputs.**

- `../data/processed/epoch/epoch_all_models.parquet`
- `../data/processed/ember/ember_yearly.parquet`
- `../data/processed/ember/ember_us_states.parquet`
- `../data/processed/im3/im3_open_source_data_center_atlas_v2026_02_09.parquet`
- `../data/processed/aqueduct/Aqueduct40_country_baseline.parquet`
- `../data/processed/aqueduct/Aqueduct40_province_baseline.parquet`
- `../data/processed/jegham/DataSnapshotOct26.parquet`
- `../data/processed/jegham/artificialanalysis_environmental.parquet`
- `../data/processed/jegham/Monthly_LLM_Environmental_Footprint.parquet`
- `../data/processed/jegham/AWS_Env_Multipliers.parquet`
- `../data/processed/jegham/Microsoft_Env_Multipliers.parquet`
- `../data/processed/README.md`

In [26]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

for source in ["epoch", "ember", "im3", "aqueduct", "jegham"]:
    (PROCESSED / source).mkdir(parents=True, exist_ok=True)

In [27]:
def validate_and_write(df, source, filename, expected_columns, expected_min_rows):
    missing = [c for c in expected_columns if c not in df.columns]
    assert not missing, f"{source}/{filename}: missing expected columns {missing}"
    assert (
        len(df) >= expected_min_rows
    ), f"{source}/{filename}: {len(df)} rows < expected min {expected_min_rows}"

    extras = [c for c in df.columns if c not in expected_columns]
    print(
        f"{source}/{filename}: rows={len(df)} cols={len(df.columns)} "
        f"extras={len(extras)}"
    )

    out = PROCESSED / source / f"{filename}.parquet"
    df.to_parquet(out, engine="pyarrow", index=False)

## Epoch

Single file: `epoch_all_models.csv`. Default `pd.read_csv`. The
`Training power draw (W)` column has documented partial coverage
(populated for ~481 post-2023 models); validation only checks that the
column is present, not how densely it is populated.

In [28]:
df_epoch = pd.read_csv(RAW / "epoch" / "epoch_all_models.csv")

validate_and_write(
    df_epoch,
    source="epoch",
    filename="epoch_all_models",
    expected_columns=[
        "Model",
        "Publication date",
        "Organization",
        "Training compute (FLOP)",
        "Parameters",
        "Training hardware",
        "Training power draw (W)",
    ],
    expected_min_rows=3000,
)

power_draw_populated = df_epoch["Training power draw (W)"].notna().sum()
print(f"epoch/epoch_all_models: Training power draw (W) populated rows = {power_draw_populated}")

epoch/epoch_all_models: rows=3509 cols=57 extras=50
epoch/epoch_all_models: Training power draw (W) populated rows = 771


## Ember

Two files in long format. Default `pd.read_csv` for both. Per the
Ember PROVENANCE, the yearly file uses `Area` + `ISO 3 code` for
country identification while the US states file uses `State` + `State
code`; reconciliation is a Phase 2 task. The `Variable` column carries
many measures (generation, demand, emissions, intensity, etc.); we do
not filter to `CO2 intensity` here.

In [29]:
df_ember_yearly = pd.read_csv(RAW / "ember" / "ember_yearly.csv")

validate_and_write(
    df_ember_yearly,
    source="ember",
    filename="ember_yearly",
    expected_columns=[
        "Area",
        "ISO 3 code",
        "Year",
        "Variable",
        "Unit",
        "Value",
    ],
    expected_min_rows=200000,
)

ember/ember_yearly: rows=371020 cols=18 extras=12


In [30]:
df_ember_states = pd.read_csv(RAW / "ember" / "ember_us_states.csv")

validate_and_write(
    df_ember_states,
    source="ember",
    filename="ember_us_states",
    expected_columns=[
        "State",
        "State code",
        "Year",
        "Variable",
        "Unit",
        "Value",
    ],
    expected_min_rows=50000,
)

ember/ember_us_states: rows=73442 cols=13 extras=7


## IM3

Single file: the IM3 open-source data center atlas. Default
`pd.read_csv`. The atlas has multiple geographic layers in the `type`
column (campus, building, etc.); we do not filter to `type == "campus"`
here — that is a Phase 2 concern.

In [31]:
df_im3 = pd.read_csv(
    RAW / "im3" / "im3_open_source_data_center_atlas_v2026_02_09.csv"
)

validate_and_write(
    df_im3,
    source="im3",
    filename="im3_open_source_data_center_atlas_v2026_02_09",
    expected_columns=[
        "id",
        "state",
        "state_abb",
        "county",
        "operator",
        "name",
        "sqft",
        "lat",
        "lon",
        "type",
    ],
    expected_min_rows=1000,
)

im3/im3_open_source_data_center_atlas_v2026_02_09: rows=1479 cols=13 extras=3


## Aqueduct

One XLSX, two sheets, two parquet outputs. Default
`pd.read_excel(path, sheet_name=...)` for both sheets — the headers sit
on row 0, and PROVENANCE confirms no preprocessing at ingestion. We do
not filter to `indicator_name == "bws"` or `weight == "Tot"`, and we do
not replace -9999 sentinels (a known Singapore artefact for the `bws`
indicator); both are Phase 2 transformations.

In [32]:
df_aq_country = pd.read_excel(
    RAW / "aqueduct" / "Aqueduct40_rankings_download_Y2023M07D05.xlsx",
    sheet_name="country_baseline",
)

validate_and_write(
    df_aq_country,
    source="aqueduct",
    filename="Aqueduct40_country_baseline",
    expected_columns=[
        "gid_0",
        "name_0",
        "indicator_name",
        "weight",
        "score",
        "score_ranked",
        "cat",
        "label",
        "un_region",
        "wb_region",
    ],
    expected_min_rows=200,
)

aqueduct/Aqueduct40_country_baseline: rows=2456 cols=10 extras=0


In [33]:
df_aq_province = pd.read_excel(
    RAW / "aqueduct" / "Aqueduct40_rankings_download_Y2023M07D05.xlsx",
    sheet_name="province_baseline",
)

validate_and_write(
    df_aq_province,
    source="aqueduct",
    filename="Aqueduct40_province_baseline",
    expected_columns=[
        "name_0",
        "name_1",
        "gid_1",
        "indicator_name",
        "weight",
        "score",
        "cat",
        "label",
    ],
    expected_min_rows=3000,
)

aqueduct/Aqueduct40_province_baseline: rows=42771 cols=12 extras=4


## Jegham

Five files. Three describe per-query environmental footprint of LLM
inference (frozen snapshot, current live snapshot, monthly time series).
Two describe hyperscaler region-level multipliers (AWS, Microsoft).
Jegham PROVENANCE describes columns in prose rather than enumerating
them; the snapshot files map cleanly to load-bearing combined-* columns,
the monthly file uses a different aggregation pattern (handled in its
own subsection), and the multipliers are validated shape-only because
the bronze-layer read leaves them with mostly unnamed columns.

In [34]:
df_jegham_snapshot = pd.read_csv(RAW / "jegham" / "DataSnapshotOct26.csv")

validate_and_write(
    df_jegham_snapshot,
    source="jegham",
    filename="DataSnapshotOct26",
    expected_columns=[
        "Model",
        "Mean Combined Energy (Wh)",
        "Mean Combined Carbon (gCO2e)",
        "Mean Combined Water (Site & Source, mL)",
    ],
    expected_min_rows=60,
)

jegham/DataSnapshotOct26: rows=194 cols=98 extras=94


In [35]:
df_jegham_aa = pd.read_csv(
    RAW / "jegham" / "artificialanalysis_environmental.csv"
)

validate_and_write(
    df_jegham_aa,
    source="jegham",
    filename="artificialanalysis_environmental",
    expected_columns=[
        "Model",
        "Mean Combined Energy (Wh)",
        "Mean Combined Carbon (gCO2e)",
        "Mean Combined Water (Site & Source, mL)",
    ],
    expected_min_rows=50,
)

jegham/artificialanalysis_environmental: rows=175 cols=103 extras=99


### Monthly footprint time series

PROVENANCE describes this file as a "monthly aggregated footprint time
series" but does not enumerate columns. The schema differs from the
two snapshot files: it uses `Avg Daily Max/Min/Average × Energy/Water/
Carbon` rather than the snapshot pattern of `Mean Combined × ...`.
Actual columns at time of ingestion (52 total): `Model`, `Query
Length`, `API ID`, `Company`, `Hardware`, `Host`, `GPUs Power Draw`,
`Non-GPUs Power Draw`, `Min/Max GPU Power Utilization`, `Non-GPU Power
Utilization`, `PUE`, `WUE (Site)`, `WUE (Source)`, `CIF`, `Size`, then
`Avg Daily {Max,Min,Average} {Energy,Water,Carbon}` with day-to-day
and intra-day standard deviations, and pooled standard deviations.
Validation checks the load-bearing subset: model identifier plus one
combined-style column for each of energy, water, and carbon (using the
`Avg Daily Average` aggregation as the combined analogue).

In [36]:
df_jegham_monthly = pd.read_csv(
    RAW / "jegham" / "Monthly_LLM_Environmental_Footprint.csv"
)

validate_and_write(
    df_jegham_monthly,
    source="jegham",
    filename="Monthly_LLM_Environmental_Footprint",
    expected_columns=[
        "Model",
        "Avg Daily Average Energy (Wh)",
        "Avg Daily Average Carbon (gCO2e)",
        "Avg Daily Average Water (mL)",
    ],
    expected_min_rows=10,
)

jegham/Monthly_LLM_Environmental_Footprint: rows=233 cols=52 extras=48


### Infrastructure multipliers

Two CSVs covering hyperscaler region-level PUE/WUE/CIF. Both are
materialised faithfully without `skiprows`. Validation is shape-only
(column count + row count); named-column validation is deferred to
Phase 2, where AWS PROVENANCE documents that `skiprows=1` is required
to expose the real headers.

In [37]:
df_jegham_aws = pd.read_csv(RAW / "jegham" / "AWS_Env_Multipliers.csv")
assert (
    df_jegham_aws.shape[1] == 7
), f"AWS_Env_Multipliers.csv: expected 7 cols, got {df_jegham_aws.shape[1]}"

# Named-column validation deferred to Phase 2 where skiprows=1 will be applied per PROVENANCE.
validate_and_write(
    df_jegham_aws,
    source="jegham",
    filename="AWS_Env_Multipliers",
    expected_columns=[],
    expected_min_rows=10,
)

jegham/AWS_Env_Multipliers: rows=12 cols=7 extras=7


In [38]:
df_jegham_microsoft = pd.read_csv(RAW / "jegham" / "Microsoft_Env_Multipliers.csv")
assert (
    df_jegham_microsoft.shape[1] == 4
), f"Microsoft_Env_Multipliers.csv: expected 4 cols, got {df_jegham_microsoft.shape[1]}"

# Named-column validation deferred to Phase 2 where skiprows=1 will be applied per PROVENANCE.
validate_and_write(
    df_jegham_microsoft,
    source="jegham",
    filename="Microsoft_Env_Multipliers",
    expected_columns=[],
    expected_min_rows=10,
)

jegham/Microsoft_Env_Multipliers: rows=28 cols=4 extras=4


In [39]:
summary_rows = []
for parquet_path in sorted(PROCESSED.glob("*/*.parquet")):
    summary_rows.append(
        {
            "source": parquet_path.parent.name,
            "file": parquet_path.name,
            "rows": len(pd.read_parquet(parquet_path)),
            "size_kb": round(parquet_path.stat().st_size / 1024, 1),
        }
    )
print(pd.DataFrame(summary_rows).to_string(index=False))

  source                                                  file   rows  size_kb
aqueduct                   Aqueduct40_country_baseline.parquet   2456     37.4
aqueduct                  Aqueduct40_province_baseline.parquet  42771    647.7
   ember                               ember_us_states.parquet  73442    637.4
   ember                                  ember_yearly.parquet 371020   1374.4
   epoch                              epoch_all_models.parquet   3509   2867.0
     im3 im3_open_source_data_center_atlas_v2026_02_09.parquet   1479     80.6
  jegham              artificialanalysis_environmental.parquet    175    173.0
  jegham                           AWS_Env_Multipliers.parquet     12      4.7
  jegham                             DataSnapshotOct26.parquet    194    181.8
  jegham                     Microsoft_Env_Multipliers.parquet     28      3.4
  jegham           Monthly_LLM_Environmental_Footprint.parquet    233    113.1
